# 로봇 음성 명령 decision 모델 학습
1. 메뉴 **런타임 → 런타임 유형 변경**에서 **T4 GPU**를 고르세요.
2. 왼쪽 **파일** 탭에 `bundle.zip`을 끌어다 놓으세요.
3. 메뉴 **런타임 → 모두 실행**을 누르세요. T4에서 수십 분 걸려요.
4. 마지막 셀이 `decision-lora.zip`을 내려받아요. 이 파일이 학습된 모델이에요.

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision "torchaudio==2.8.0" bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!unsloth install-kernels
!uv pip install --no-deps --upgrade "torchao>=0.16.0"
!uv pip install kiwipiepy==0.24.1

In [ ]:
from pathlib import Path
from zipfile import ZipFile

bundle = Path("/content/bundle.zip")
if not bundle.is_file():
    raise FileNotFoundError("왼쪽 파일 탭에 bundle.zip을 올린 뒤 다시 실행하세요.")
with ZipFile(bundle) as archive:
    archive.extractall("/content")

import os
os.chdir("/content")
import pipeline

In [ ]:
MODEL = "unsloth/Qwen3.5-0.8B"
MAX_SEQ_LENGTH = 1024
LOAD_IN_4BIT = True
MAX_STEPS = -1
BATCH = 8
GRAD_ACCUM = 2
LR = 2e-4

In [ ]:
from unsloth import FastDecisionModel, DecisionTrainer, is_bfloat16_supported
import torch

model, tokenizer = FastDecisionModel.from_pretrained(
    model_name = MODEL,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = LOAD_IN_4BIT,
)
model = FastDecisionModel.get_peft_model(
    model,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files = {
    "train": "/content/train.jsonl",
    "val": "/content/val.jsonl",
})
train_items, train_report = FastDecisionModel.build_dataset(dataset["train"], tokenizer, model)
val_items, val_report = FastDecisionModel.build_dataset(dataset["val"], tokenizer, model)
print(f"train: {train_report['total']}개 중 {train_report['skipped']}개 제외")
print(f"val: {val_report['total']}개 중 {val_report['skipped']}개 제외")
assert train_items and val_items, "학습 또는 검증 데이터가 비어 있습니다."

In [ ]:
from transformers import TrainingArguments

trainer = DecisionTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = train_items,
    eval_dataset = val_items,
    args = TrainingArguments(
        per_device_train_batch_size = BATCH,
        gradient_accumulation_steps = GRAD_ACCUM,
        warmup_steps = 20,
        num_train_epochs = 1,
        max_steps = MAX_STEPS if MAX_STEPS > 0 else -1,
        learning_rate = LR,
        lr_scheduler_type = "cosine",
        weight_decay = 0.01,
        logging_steps = 25,
        bf16 = is_bfloat16_supported(),
        fp16 = not is_bfloat16_supported(),
        save_strategy = "no",
        report_to = "none",
        eval_strategy = "epoch",
        output_dir = "outputs",
        seed = 3407,
    ),
)
trainer_stats = trainer.train()

In [ ]:
print(FastDecisionModel.calibrate(model, tokenizer, val_items))
print(FastDecisionModel.evaluate(model, tokenizer, val_items))

## 실제 녹음 33개로 전체 평가

In [ ]:
import json
from time import perf_counter

FastDecisionModel.for_inference(model)
real_rows = [json.loads(line) for line in Path("/content/eval_real.jsonl").read_text(encoding="utf-8").splitlines()]
assert len(real_rows) == 33, "실제 녹음 평가 데이터는 33개여야 합니다."
matches = 0
utterance_seconds, segment_seconds = [], []
for row in real_rows:
    began = perf_counter()
    decisions = []
    for segment, state in pipeline.windows(pipeline.normalize_numbers(row["asr"])):
        torch.cuda.synchronize()
        start = perf_counter()
        answers = FastDecisionModel.predict(model, tokenizer, state, pipeline.QUESTIONS)
        torch.cuda.synchronize()
        segment_seconds.append(perf_counter() - start)
        decisions.append((
            segment, answers["robot"]["answer"], answers["action"]["answer"],
            answers["direction"]["answer"],
        ))
    commands = pipeline.assemble(decisions)
    utterance_seconds.append(perf_counter() - began)
    if commands == row["commands"]:
        matches += 1
        print(row["id"], "OK")
    else:
        print(row["id"], "NG", "예측:", commands, "정답:", row["commands"])

print(f"명령 목록 완전 일치: {matches}/{len(real_rows)}")
print(f"발화당 평균: {sum(utterance_seconds) / len(real_rows):.3f}초 (ASR 제외)")
print(f"조각당 predict 평균: {sum(segment_seconds) / len(segment_seconds):.3f}초")

## 학습된 모델 저장

In [ ]:
import shutil
from google.colab import files

model.save_pretrained("decision-lora")
shutil.make_archive("decision-lora", "zip", ".", "decision-lora")
files.download("decision-lora.zip")